# Huggingface pipeline

Install all required packages for the fine-tuning experiment.

In [1]:
import importlib.metadata
import subprocess
import sys

def install_dep(modules: list):
    for pkg in modules:
        base = pkg.split("[")[0]  # strip extras like [torch]
        try:
            importlib.metadata.distribution(base)
            print(f"{pkg} already installed")
        except importlib.metadata.PackageNotFoundError:
            print(f"Installing {pkg}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

modules = ["huggingface_hub",
           "datasets", 
           "transformers", 
           "torch", 
           "numpy", 
           "nvidia-ml-py", 
           "transformers[torch]", 
           "evaluate",
           "ipywidgets",
           "scikit-learn",
           "scipy",
           "accelerate"]
install_dep(modules)

huggingface_hub already installed
datasets already installed
transformers already installed
torch already installed
numpy already installed
nvidia-ml-py already installed
transformers[torch] already installed
evaluate already installed
ipywidgets already installed
scikit-learn already installed
scipy already installed
accelerate already installed


Login with notebook to Huggingface

In [2]:
from huggingface_hub import notebook_login
notebook_login()

Load the MRPC dataset from GLUE — 3 668 training and 408 validation sentence pairs labeled as paraphrase or not.

In [3]:
from datasets import load_dataset

raw_datasets = load_dataset("nyu-mll/glue", "mrpc")
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

Load the BERT tokenizer, define a tokenisation function for sentence pairs, and create a `DataCollatorWithPadding` for dynamic batching.

In [4]:
from transformers import AutoTokenizer, DataCollatorWithPadding
checkpoint = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenizer_function(item):
    return tokenizer(item["sentence1"],item["sentence2"],truncation=True,padding="max_length",return_tensors="pt")
data_collactor = DataCollatorWithPadding(tokenizer=tokenizer)

Apply tokenisation to train and validation sets; drop raw text columns, rename `label` → `labels`, and format as PyTorch tensors.

In [5]:
train_dataset = raw_datasets["train"]
train_dataset = train_dataset.map(tokenizer_function,batched=True)
train_dataset = train_dataset.remove_columns(["sentence1","sentence2","idx"])
train_dataset = train_dataset.rename_column("label","labels")
train_dataset.set_format("torch")

validation_dataset = raw_datasets["validation"]
validation_dataset = validation_dataset.map(tokenizer_function,batched=True)
validation_dataset = validation_dataset.remove_columns(["sentence1","sentence2","idx"])
validation_dataset = validation_dataset.rename_column("label","labels")
validation_dataset.set_format("torch")


Load pretrained `bert-base-uncased` with a fresh 2-class classification head (the MLM/NSP head weights are discarded).

In [6]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint,num_labels=2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Wrap the tokenized train and validation datasets in PyTorch `DataLoader`s so they can be iterated in batches during manual training. The training loader shuffles examples each epoch with a batch size of 8, while the validation loader keeps the original order since shuffling isn't needed for evaluation. Both loaders use the `DataCollatorWithPadding` collator defined earlier, which pads each batch dynamically to the length of its longest sequence rather than a fixed maximum. This keeps batches as small as possible and speeds up training compared to padding everything to `max_length` up front. From here on, training is driven manually instead of through the high-level `Trainer` API.

In [7]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader (train_dataset, shuffle=True, batch_size=8,collate_fn=data_collactor)
validation_dataloader = DataLoader (validation_dataset, batch_size=8,collate_fn=data_collactor)

Sanity-check the model and data pipeline before committing to a full training run. A single batch is pulled from the validation dataloader using a `for`/`break` pattern, then fed through the model's `forward` method with gradient tracking disabled via `torch.no_grad()`. Because the batch still contains the `labels` field, the model computes and returns a loss alongside the raw logits, confirming the loss function is wired up correctly. This is purely a smoke test — no weights are updated — and is useful for catching shape mismatches or device errors early, before they show up mid-training.

In [8]:
import torch
for batch in validation_dataloader:
    break
with torch.no_grad():
    prediction = model.forward(**batch)
prediction

SequenceClassifierOutput(loss=tensor(0.7655), logits=tensor([[ 0.3841, -0.3916],
        [ 0.3370, -0.3979],
        [ 0.3510, -0.3790],
        [ 0.3723, -0.3906],
        [ 0.3573, -0.3788],
        [ 0.3472, -0.3831],
        [ 0.3758, -0.3944],
        [ 0.3683, -0.3920]]), hidden_states=None, attentions=None)

Create the optimizer that will update the model's weights during manual training. `AdamW` is the standard choice for fine-tuning transformer models, since it decouples weight decay from the gradient-based update in a way that works better than plain Adam for these architectures. It's initialized with all of the model's parameters and a learning rate of `5e-5`, a typical starting point for BERT-style fine-tuning. No training happens in this cell — it only sets up the optimizer object that the manual training loop will call `.step()` and `.zero_grad()` on.

In [9]:
# Load the optimizer
from torch.optim import AdamW

optimizer = AdamW(model.parameters(),lr=5e-5)

Set up Hugging Face `Accelerate` and run the actual manual training loop. An `Accelerator` instance wraps the dataloaders, model, and optimizer via `accelerator.prepare()`, which transparently handles moving everything to the right device (and would handle multi-GPU/mixed precision if configured), removing the need for manual `.to(device)` calls. A linear learning-rate scheduler is created for 3 epochs of training with no warmup steps, and a `tqdm` progress bar tracks the total number of optimization steps. The training loop itself is fully manual: for each batch it runs a forward pass, computes the loss, calls `accelerator.backward(loss)` instead of `loss.backward()` so Accelerate can handle gradient scaling/distribution correctly, then steps the optimizer and scheduler before zeroing gradients. This cell is the core training step that the earlier dataloader, model, and optimizer cells were all building toward.

In [10]:
from accelerate import Accelerator, notebook_launcher
from tqdm.auto import tqdm
from transformers import get_scheduler

def train_with_accelerator():
    accelerator = Accelerator()
    train_dl, eval_dl, model, optimizer = accelerator.prepare(
        train_dataloader, validation_dataloader, model, optimizer
    )
    num_epochs = 3
    num_training_steps = num_epochs * len(train_dl)
    lr_scheduler = get_scheduler(
        "linear",
        optimizer=optimizer,
        num_warmup_steps=0,
        num_training_steps=num_training_steps,
    )
    progress_bar = tqdm(range(num_training_steps))
    print(num_training_steps)

    model.train()
    for epoch in range(num_epochs):
        for batch in train_dl:
            outputs = model(**batch)
            loss = outputs.loss
            accelerator.backward(loss)

            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()
            progress_bar.update(1)

notebook_launcher(train_with_accelerator,num_processes=2)

Launching training on 2 CUDAs.


E0630 22:36:16.737000 514052 torch/distributed/elastic/multiprocessing/api.py:836] failed (exitcode: 1) local_rank: 0 (pid: 514414) of fn: train_with_accelerator (start_method: fork)
E0630 22:36:16.737000 514052 torch/distributed/elastic/multiprocessing/api.py:836] Traceback (most recent call last):
E0630 22:36:16.737000 514052 torch/distributed/elastic/multiprocessing/api.py:836]   File "/home/user/git/rigorous-ai/.venv/lib/python3.12/site-packages/torch/distributed/elastic/multiprocessing/api.py", line 791, in _poll
E0630 22:36:16.737000 514052 torch/distributed/elastic/multiprocessing/api.py:836]     self._pc.join(-1)
E0630 22:36:16.737000 514052 torch/distributed/elastic/multiprocessing/api.py:836]   File "/home/user/git/rigorous-ai/.venv/lib/python3.12/site-packages/torch/multiprocessing/spawn.py", line 211, in join
E0630 22:36:16.737000 514052 torch/distributed/elastic/multiprocessing/api.py:836]     raise ProcessRaisedException(msg, error_index, failed_process.pid)
E0630 22:36:1

ChildFailedError: 
============================================================
train_with_accelerator FAILED
------------------------------------------------------------
Failures:
  <NO_OTHER_FAILURES>
------------------------------------------------------------
Root Cause (first observed failure):
[0]:
  time      : 2026-06-30_22:36:16
  host      : linux
  rank      : 0 (local_rank: 0)
  exitcode  : 1 (pid: 514414) 
  error_file: /tmp/torchelastic_sup7nol6/none_gxka_6c1/attempt_0/0/error.json
  traceback : Traceback (most recent call last):
    File "/home/user/git/rigorous-ai/.venv/lib/python3.12/site-packages/torch/distributed/elastic/multiprocessing/errors/__init__.py", line 367, in wrapper
      return f(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^
    File "/tmp/ipykernel_514052/1191925906.py", line 8, in train_with_accelerator
      train_dataloader, validation_dataloader, model, optimizer
                                               ^^^^^
  UnboundLocalError: cannot acc

Evaluate the fine-tuned model on the MRPC validation set using the official GLUE metric. The `evaluate` library loads the MRPC metric, which reports both accuracy and F1 score — the same metrics used on the GLUE leaderboard for this task. The model is switched to evaluation mode with `model.eval()`, and predictions are accumulated batch by batch under `torch.no_grad()` since no gradients are needed at inference time. For each batch, logits are converted to class predictions via `argmax`, and `metric.add_batch()` collects them alongside the true labels for the final `metric.compute()` call. The resulting scores indicate how well the manually trained model generalizes to unseen paraphrase pairs.

In [ ]:
import evaluate

metric = evaluate.load("glue", "mrpc")
model.eval()
for batch in eval_dl:
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(**batch)

    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)
    metric.add_batch(predictions=predictions, references=batch["labels"])

metric.compute()

{'accuracy': 0.6838235294117647, 'f1': 0.8122270742358079}